# 🔌 Spain EV Charging Infrastructure Analysis
### Identifying Optimal Locations for New Fast-Charging Stations

---

## Overview

This notebook builds a **data-driven framework** for evaluating where new EV charging infrastructure should be deployed along Spain's interurban road network.

| Layer | Source | Purpose |
|---|---|---|
| Road network | OpenStreetMap (Overpass API) | Structural backbone of mobility corridors |
| Traffic flows | Ministry of Transport – BigData Movilidad | Demand signal (travellers/day per segment) |
| EV charging stations | DGT DATEX II XML feed | Existing supply (location, connectors, power) |
| Power grid access | i-DE (Iberdrola) CNMC demand data | Grid viability for new fast chargers |

**Regulatory context:** EU AFIR Regulation 2023/1804 mandates HPC chargers every 60 km on TEN-T Core corridors and every 100 km on Comprehensive corridors by 2025–2030.

---

## Navigation

1. [Setup & Imports](#setup)
2. [Road Network & Traffic](#roads)
3. [EV Charging Stations](#chargers)
4. [Power Grid Access](#grid)
5. [Combined Analysis Map](#combined)


---
<a id="setup"></a>
## 0. Setup & Imports

All dependencies are loaded here. The notebook uses:
- **GeoPandas / Shapely** for spatial operations
- **Folium** for interactive maps
- **Requests** for API calls to OSM Overpass and Ministry of Transport

> **Data sources:**
> - OpenStreetMap — roads, CC BY-SA
> - Ministry of Transport BigData Movilidad — traffic flows
> - DGT DATEX II — charging station registry
> - i-DE / CNMC — grid access points


In [1]:
from IPython.display import display, HTML
display(HTML("<h3>Notebook is now trusted (reload page)</h3>"))

In [2]:
# Standard library
import io, os, json, math, warnings
import numpy as np
import pandas as pd
import re, requests, time
from shapely.geometry import LineString, MultiLineString, Point
from shapely.ops import unary_union

warnings.filterwarnings('ignore')

# Geospatial
import geopandas as gpd
import pyproj

# Visualisation
import folium
from folium.plugins import HeatMap, MarkerCluster
from IPython.display import IFrame, display

# ── Global constants ──────────────────────────────────────────────────────────
CRS_GEO    = 'EPSG:4326'    # WGS84 — all final outputs must use this
CRS_METRIC = 'EPSG:25830'   # ETRS89/UTM30N — for accurate km calculations
MAX_GAP_KM = 150            # Max allowed gap between chargers (AFIR regulation)
CHARGER_KW = 150            # Fixed charger power — datathon rules
MIN_CHARGERS = 2            # AFIR minimum chargers per station
MAX_CHARGERS = 12           # Practical cap (space + demand)
COVERAGE_BUFFER_M = 50_000  # 50 km coverage radius per existing charger

# ── Map defaults ──────────────────────────────────────────────────────────────
SPAIN_CENTER = [40.2, -3.7]     # Approx geographic centre of mainland Spain
DEFAULT_ZOOM = 6
MAP_TILES    = "CartoDB positron"  # Lightweight basemap, good for overlays

from pathlib import Path
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

print("Setup complete")


Setup complete


---
<a id="roads"></a>
## 1. Road Network & Traffic Analysis

This section integrates Spain's interurban road network with observed traffic flows to identify where **mobility demand is highest**.

The goal is to move from a purely structural view of roads to a demand-driven understanding of the network — essential for optimally placing EV charging infrastructure.

### Approach
- Road network (OSM) provides geographic context and routing geometry.
- Traffic data (Ministry of Transport) provides segment-level daily traveller estimates.
- Traffic is mapped directly onto road segments, preserving spatial variation along the network.

### Assumptions
- `Total` is used as the proxy for traffic intensity and potential charging demand.
- Filtering by road codes `(AP, A, N, R, M)` approximates interurban roads where EV charging demand is most relevant.
- Geometry simplification is applied **for visualisation only** — it does not affect distance calculations.


### 1.1 Road Network (OpenStreetMap)

Fetches interurban road geometries via the Overpass API. Two mirrors are tried in order.

**Filter:** Road codes matching `AP-`, `A-`, `N-`, `R-`, `M-` prefixes (motorways, national roads, ring roads).

> Source: OpenStreetMap contributors, ODbL licence.


In [4]:
QUERY_ROADS = """
[out:json][timeout:120][bbox:35.9,-9.5,43.8,4.5];
(
  way["highway"]["ref"~"^(AP-|A-|N-|R-|M-)"];
);
out geom;
"""

OVERPASS_MIRRORS = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://maps.mail.ru/osm/tools/overpass/api/interpreter',
    'https://overpass.openstreetmap.ru/api/interpreter',
]

HEADERS = {
    'User-Agent': 'IberdrolaEVResearch/1.0 (datathon academic project)',
    'Accept': 'application/json',
}

print('Fetching road network from OpenStreetMap...')
raw = None
for url in OVERPASS_MIRRORS:
    # Try POST first, fall back to GET (some mirrors prefer one over the other)
    for method in ('post', 'get'):
        try:
            if method == 'post':
                resp = requests.post(url, data={'data': QUERY_ROADS},
                                     headers=HEADERS, timeout=120)
            else:
                resp = requests.get(url, params={'data': QUERY_ROADS},
                                    headers=HEADERS, timeout=120)
            resp.raise_for_status()
            raw = resp.json()
            print(f'{url.split("/")[2]} ({method.upper()}) — {len(raw["elements"]):,} elements')
            break
        except Exception as e:
            print(f'{url.split("/")[2]} {method.upper()} failed: {e}')
    if raw is not None:
        break

if raw is None:
    raise RuntimeError('All Overpass mirrors failed.')

# Parse JSON into rows
rows = []
for el in raw['elements']:
    if 'geometry' not in el: continue
    coords = [(p['lon'], p['lat']) for p in el['geometry']]
    if len(coords) < 2: continue
    t = el.get('tags', {})
    ref = (t.get('ref') or '').split(';')[0].strip().upper()
    rows.append({
        'ref': ref,
        'highway': t.get('highway', ''),
        'name': t.get('name', ''),
        'geometry': LineString(coords),
    })

roads = gpd.GeoDataFrame(rows, geometry='geometry', crs=CRS_GEO)

print(f'\nRoads loaded: {len(roads):,} segments')
print('Top road codes:')
print(roads['ref'].value_counts().head(10).to_string())


Fetching road network from OpenStreetMap...
overpass-api.de (POST) — 148,609 elements

Roads loaded: 148,609 segments
Top road codes:
ref
N-340     3563
A-7       3446
N-634     3040
N-II      2361
N-260     2339
N-630     2301
AP-7      2079
N-240     1836
N-340A    1811
N-6       1666


### 1.2 Traffic Data (Ministry of Transport)

Fetches segment-level daily traveller estimates from the Ministry of Transport's ArcGIS REST API.
Results are paginated in batches of 2,000 and CRS-converted from EPSG:3042 to WGS84.

**Field used:** `Total` — total daily travellers across all trip lengths.

> Source: Ministerio de Transportes, Movilidad y Agenda Urbana — BigData Movilidad 2.


In [5]:
URL = "https://mapas.fomento.gob.es/arcgis2/rest/services/BigData/Movilidad_Big_Data_2/MapServer/1282/query"

BATCH_SIZE = 2000
offset = 0
rows = []

def arcgis_polyline_to_shapely(geom):
    # Convert ArcGIS JSON polyline paths to a Shapely geometry
    paths = geom.get("paths", [])
    if not paths:
        return None
    line_parts = [LineString(path) for path in paths if len(path) >= 2]
    if not line_parts:
        return None
    return line_parts[0] if len(line_parts) == 1 else MultiLineString(line_parts)

print("Fetching official traffic layer...")

while True:
    params = {
        "f": "json",
        "where": "1=1",
        "returnGeometry": "true",
        "outFields": "*",
        "orderByFields": "OBJECTID ASC",
        "resultOffset": offset,
        "resultRecordCount": BATCH_SIZE
        # no outSR: keep native layer geometry (EPSG:3042)
    }

    r = requests.get(URL, params=params, timeout=120)
    r.raise_for_status()
    data = r.json()

    features = data.get("features", [])
    if not features:
        break

    kept = 0
    for f in features:
        attrs = f.get("attributes", {}).copy()
        shp = arcgis_polyline_to_shapely(f.get("geometry", {}))
        if shp is None:
            continue
        attrs["geometry"] = shp
        rows.append(attrs)
        kept += 1

    print(f"Offset {offset:,}: kept {kept:,} features; total {len(rows):,}")

    if len(features) < BATCH_SIZE:
        break

    offset += BATCH_SIZE
    time.sleep(0.2)

traffic_gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:3042").to_crs(4326)

print(traffic_gdf.shape)
traffic_gdf.head()


Fetching official traffic layer...
Offset 0: kept 2,000 features; total 2,000
Offset 2,000: kept 2,000 features; total 4,000
Offset 4,000: kept 2,000 features; total 6,000
Offset 6,000: kept 2,000 features; total 8,000
Offset 8,000: kept 2,000 features; total 10,000
Offset 10,000: kept 2,000 features; total 12,000
Offset 12,000: kept 2,000 features; total 14,000
Offset 14,000: kept 2,000 features; total 16,000
Offset 16,000: kept 2,000 features; total 18,000
Offset 18,000: kept 2,000 features; total 20,000
Offset 20,000: kept 2,000 features; total 22,000
Offset 22,000: kept 2,000 features; total 24,000
Offset 24,000: kept 2,000 features; total 26,000
Offset 26,000: kept 2,000 features; total 28,000
Offset 28,000: kept 2,000 features; total 30,000
Offset 30,000: kept 2,000 features; total 32,000
Offset 32,000: kept 2,000 features; total 34,000
Offset 34,000: kept 2,000 features; total 36,000
Offset 36,000: kept 2,000 features; total 38,000
Offset 38,000: kept 2,000 features; total 40,00

,OBJECTID,nombre,Total,Corto,p_Corto,Medio,p_Medio,Largo,p_Largo,Intra_GAU,...,Intra_ccaa,Inter_ccaa,p_IntraCA,Nacional,Extranjero,Fecha_dato,Valido_desde,Shape_Length,Valido_hasta,geometry
0,1,CV-804,817.04,793.42,97,23.62,3,0.00,0,0.00,...,813.77,3.27,100,817.04,0.0,1729036800000,1748217600000,85.607315,None,"LINESTRING (-0.67046 38.71026, -0.67021 38.710..."
1,2,CV-700,1660.57,1501.45,90,152.74,9,6.38,0,206.57,...,1637.62,22.95,99,1660.57,0.0,1729036800000,1748217600000,93.262908,None,"LINESTRING (-0.45157 38.78392, -0.45061 38.78355)"
2,3,N-301,92.02,73.94,80,11.22,12,6.86,7,51.03,...,77.15,14.87,84,92.02,0.0,1729036800000,1748217600000,22.603400,None,"LINESTRING (-1.69968 38.54366, -1.69942 38.54366)"
3,4,CV-799,2853.52,2401.01,84,445.96,16,6.55,0,9.27,...,2799.14,54.38,98,2853.52,0.0,1729036800000,1748217600000,356.019092,None,"LINESTRING (-0.76098 38.62795, -0.76099 38.627..."
4,5,N-301,152.61,115.21,75,29.38,19,8.02,5,44.65,...,146.53,6.08,96,152.61,0.0,1729036800000,1748217600000,24.250378,None,"LINESTRING (-1.69917 38.54352, -1.69937 38.543..."


In [6]:
# ── Type coercion: cast numeric columns and parse date fields ─────────────────
numeric_cols = [
    "Total", "Corto", "Medio", "Largo",
    "Intra_GAU", "Inter_GAU", "Intra_prov", "Inter_prov",
    "Intra_ccaa", "Inter_ccaa", "Nacional", "Extranjero", "Shape_Length"
]

for col in numeric_cols:
    if col in traffic_gdf.columns:
        traffic_gdf[col] = pd.to_numeric(traffic_gdf[col], errors="coerce")

# Normalise road reference to match OSM road codes
traffic_gdf["ref"] = (
    traffic_gdf["nombre"].astype(str).str.strip().str.upper()
)

for col in ["Fecha_dato", "Valido_desde", "Valido_hasta"]:
    if col in traffic_gdf.columns:
        traffic_gdf[col] = pd.to_datetime(traffic_gdf[col], unit="ms", errors="coerce")

# Drop rows without a traffic value or geometry
traffic_gdf = traffic_gdf[
    traffic_gdf["Total"].notna() & traffic_gdf.geometry.notna()
].copy()

print("Traffic features:", len(traffic_gdf))
print("Unique road names:", traffic_gdf["nombre"].nunique())


Traffic features: 406626
Unique road names: 4116


In [7]:
# ── Filter to interurban road codes only ──────────────────────────────────────
# Keeps AP- (toll motorways), A- (free motorways), N- (national), R-, M- (Madrid ring)
import re as _re
keep_pattern = r"^(AP-\d+[A-Z]*|A-\d+[A-Z]*|N-\d+[A-Z]*|R-\d+[A-Z]*|M-\d+[A-Z]*)$"

traffic_gdf = traffic_gdf[
    traffic_gdf["ref"].str.match(keep_pattern, na=False)
].copy()

print("After road-code filter:", len(traffic_gdf))


After road-code filter: 242102


In [8]:
# ── Prepare lightweight copy for map rendering ────────────────────────────────
# Simplify geometries for performance — does NOT affect analysis columns
traffic_map = traffic_gdf.to_crs(3042).copy()
traffic_map["seg_len_m"] = traffic_map.geometry.length
traffic_map = traffic_map[traffic_map["seg_len_m"] >= 20].copy()  # drop sub-20m artefacts
traffic_map["geometry"] = traffic_map.geometry.simplify(10, preserve_topology=True)
traffic_map = traffic_map.drop(columns=["seg_len_m"]).to_crs(4326)

# ── Background road layer: OSM roads simplified for map rendering ─────────────
roads_bg = roads[["ref", "name", "highway", "geometry"]].copy().to_crs(3857)
roads_bg["seg_len_m"] = roads_bg.geometry.length
roads_bg = roads_bg[roads_bg["seg_len_m"] >= 150].copy()
roads_bg["geometry"] = roads_bg.geometry.simplify(80, preserve_topology=True)
roads_bg = roads_bg.drop(columns="seg_len_m").to_crs(4326)

print("Map-ready traffic features:", len(traffic_map))
print("Background road segments:", len(roads_bg))


Map-ready traffic features: 201983
Background road segments: 68040


### Map 1 — Road Network & Traffic Intensity

The map below shows the interurban road network coloured by **daily traffic intensity**.

**Colour scale (travellers/day):**
- Blue — < 5,000 (low demand)
- Green — 5,000–20,000
- Orange — 20,000–50,000
- Red — > 50,000 (high demand corridors)
- Grey — structural reference (no matched traffic data)

> Red and orange segments are priority corridors for charging deployment.
> Low-traffic blue segments may still require coverage to satisfy the AFIR 150 km gap rule.


---
<a id="chargers"></a>
## 2. EV Charging Stations

This section loads the **official Spanish EV charging station registry** published by the DGT in DATEX II XML format.

Each station (*site*) contains one or more *refill points*, each with one or more *connectors*. Parsed into:
- **`sites_df`** — one row per station (location, operator, power summary)
- **`connectors_df`** — one row per connector (type, format, max power)

> **Source:** DGT DATEX II v3 XML feed — updated daily.

### Limitations
- Power values are self-reported by operators and may be inconsistent.
- Stations without GPS coordinates are excluded from spatial analysis.
- Connector count reflects hardware capacity, not operational availability.


In [9]:
import xml.etree.ElementTree as ET

url = "https://infocar.dgt.es/datex2/v3/miterd/EnergyInfrastructureTablePublication/electrolineras.xml"

r = requests.get(url, timeout=60)
r.raise_for_status()

root = ET.fromstring(r.content)
print("Root tag:", root.tag)


Root tag: {http://datex2.eu/schema/3/d2Payload}payload


In [10]:
# ── XML helper functions ──────────────────────────────────────────────────────

def strip_ns(tag):
    # Remove XML namespace prefix, e.g. "{http://...}latitude" → "latitude"
    return tag.split("}", 1)[-1]

def find_child_text(elem, path_tags):
    # Walk down a sequence of child tags and return the text of the final node.
    # Namespace-agnostic: works regardless of XML namespace declarations.
    cur = elem
    for tag in path_tags:
        nxt = None
        for child in cur:
            if strip_ns(child.tag) == tag:
                nxt = child
                break
        if nxt is None:
            return None
        cur = nxt
    return (cur.text or "").strip() if cur.text else None

def extract_coordinates(site):
    # Robustly find latitude/longitude anywhere inside a site element
    lat = lon = None
    for el in site.iter():
        tag = strip_ns(el.tag)
        txt = (el.text or "").strip()
        if tag == "latitude" and txt:
            try: lat = float(txt)
            except: pass
        elif tag == "longitude" and txt:
            try: lon = float(txt)
            except: pass
    return lat, lon

def safe_float(x):
    try: return float(x) if x not in [None, ""] else None
    except: return None

def extract_address_parts(site):
    # Extract structured address components from addressLine entries
    result = {"street": None, "municipality": None, "province": None,
              "autonomous_community": None, "postcode": None}

    result["postcode"] = find_child_text(
        site, ["locationReference", "_locationReferenceExtension",
               "facilityLocation", "address", "postcode"])

    for addr in site.iter():
        if strip_ns(addr.tag) != "addressLine":
            continue
        line_value = None
        for child in addr:
            if strip_ns(child.tag) == "text":
                line_value = find_child_text(child, ["values", "value"])
        if not line_value:
            continue
        low = line_value.lower()
        if low.startswith("dirección:") or low.startswith("direccion:"):
            result["street"] = line_value.split(":", 1)[1].strip()
        elif low.startswith("municipio:"):
            result["municipality"] = line_value.split(":", 1)[1].strip()
        elif low.startswith("provincia:"):
            result["province"] = line_value.split(":", 1)[1].strip()
        elif low.startswith("comunidad autónoma:") or low.startswith("comunidad autonoma:"):
            result["autonomous_community"] = line_value.split(":", 1)[1].strip()
    return result


In [11]:
# ── Parse XML into site and connector tables ──────────────────────────────────
site_rows = []
connector_rows = []

sites = [el for el in root.iter() if strip_ns(el.tag) == "energyInfrastructureSite"]
print(f"Sites found: {len(sites):,}")

for site_idx, site in enumerate(sites, start=1):
    site_name     = find_child_text(site, ["name", "values", "value"])
    last_updated  = find_child_text(site, ["lastUpdated"])
    operator_name = find_child_text(site, ["operator", "name", "values", "value"])
    type_of_site  = find_child_text(site, ["typeOfSite"])
    latitude, longitude = extract_coordinates(site)
    addr = extract_address_parts(site)

    refill_points = [el for el in site.iter() if strip_ns(el.tag) == "refillPoint"]
    site_connector_count = 0
    site_max_power = None

    for rp_idx, rp in enumerate(refill_points, start=1):
        rp_name    = find_child_text(rp, ["name", "values", "value"])
        connectors = [el for el in rp if strip_ns(el.tag) == "connector"]

        for conn_idx, conn in enumerate(connectors, start=1):
            connector_type   = find_child_text(conn, ["connectorType"])
            charging_mode    = find_child_text(conn, ["chargingMode"])
            connector_format = find_child_text(conn, ["connectorFormat"])
            max_power        = safe_float(find_child_text(conn, ["maxPowerAtSocket"]))
            maximum_current  = safe_float(find_child_text(conn, ["maximumCurrent"]))

            site_connector_count += 1
            if max_power is not None:
                site_max_power = max(site_max_power, max_power) if site_max_power else max_power

            connector_rows.append({
                "site_id": site_idx, "site_name": site_name,
                "operator": operator_name, "postcode": addr["postcode"],
                "street": addr["street"], "municipality": addr["municipality"],
                "province": addr["province"],
                "autonomous_community": addr["autonomous_community"],
                "latitude": latitude, "longitude": longitude,
                "type_of_site": type_of_site, "last_updated": last_updated,
                "refill_point_id": rp_idx, "refill_point_name": rp_name,
                "connector_id": conn_idx, "connector_type": connector_type,
                "charging_mode": charging_mode, "connector_format": connector_format,
                "max_power_kw": max_power, "maximum_current_a": maximum_current,
            })

    site_rows.append({
        "site_id": site_idx, "site_name": site_name, "operator": operator_name,
        "postcode": addr["postcode"], "street": addr["street"],
        "municipality": addr["municipality"], "province": addr["province"],
        "autonomous_community": addr["autonomous_community"],
        "latitude": latitude, "longitude": longitude,
        "type_of_site": type_of_site, "last_updated": last_updated,
        "n_refill_points": len(refill_points),
        "n_connectors": site_connector_count,
        "site_max_power_kw": site_max_power,
    })

sites_df      = pd.DataFrame(site_rows)
connectors_df = pd.DataFrame(connector_rows)

print("Sites table:", sites_df.shape)
print("Connectors table:", connectors_df.shape)
sites_df.head()


Sites found: 12,075
Sites table: (12075, 15)
Connectors table: (42741, 20)


,site_id,site_name,operator,postcode,street,municipality,province,autonomous_community,latitude,longitude,type_of_site,last_updated,n_refill_points,n_connectors,site_max_power_kw
0,1,QWELLO - Calle Juan Antonio Zenón 90,QWELLO España SL,28600,Calle Juan Antonio Zenón 90,Navalcarnero,Madrid,"Madrid, Comunidad de",40.288380,-4.020607,openSpace,2026-04-13T00:33:58.000+02:00,2,2,22000.0
1,2,Petrem Eco Moli,PETREM DISTRIBUCIO SA,17600,Av. Vilallonga 61,Figueres,Girona,Cataluña,42.266670,2.973634,openSpace,2026-04-13T00:33:48.000+02:00,1,2,50000.0
2,3,PETRO UVE LAVADERO,PETRO UVE S.L,41740,AVENIDA NERVION 61,Lebrija,Sevilla,Andalucía,36.911285,-6.083734,openSpace,2026-04-08T00:47:41.000+02:00,1,1,40000.0
3,4,PARKING CENTRO,ARSA INVERSIONES SAU,21002,CALLE LOS CALIFAS Nº 2,Huelva,Huelva,Andalucía,37.258450,-6.957360,openSpace,2026-04-07T11:44:41.000+02:00,3,3,22000.0
4,5,ESTACION SERVICIO MIRALBUENO S.L,ESTACION SERVICIO MIRALBUENO S.L,50011,CARRETERA LOGROÑO NUMERO 42,Zaragoza,Zaragoza,Aragón,41.663418,-0.930423,onstreet,2026-03-31T09:13:00.000+02:00,1,4,180000.0


In [12]:
# ── Aggregated summaries ──────────────────────────────────────────────────────

summary_province = (
    connectors_df.groupby("province", dropna=False)
    .agg(total_connectors=("connector_id","count"), total_sites=("site_id","nunique"),
         total_operators=("operator","nunique"), max_power_kw=("max_power_kw","max"),
         avg_power_kw=("max_power_kw","mean"))
    .reset_index().sort_values("total_connectors", ascending=False)
)

summary_municipality = (
    connectors_df.groupby(["province","municipality"], dropna=False)
    .agg(total_connectors=("connector_id","count"), total_sites=("site_id","nunique"),
         total_operators=("operator","nunique"), max_power_kw=("max_power_kw","max"),
         avg_power_kw=("max_power_kw","mean"))
    .reset_index().sort_values("total_sites", ascending=False)
)

summary_autonomous_community = (
    connectors_df.groupby("autonomous_community", dropna=False)
    .agg(total_connectors=("connector_id","count"), total_sites=("site_id","nunique"),
         total_operators=("operator","nunique"), max_power_kw=("max_power_kw","max"),
         avg_power_kw=("max_power_kw","mean"))
    .reset_index().sort_values("total_sites", ascending=False)
)

summary_operator = (
    connectors_df.groupby("operator", dropna=False)
    .agg(total_connectors=("connector_id","count"), total_sites=("site_id","nunique"),
         provinces_covered=("province","nunique"), municipalities_covered=("municipality","nunique"),
         max_power_kw=("max_power_kw","max"), avg_power_kw=("max_power_kw","mean"))
    .reset_index().sort_values("total_connectors", ascending=False)
)

summary_operator_connector_type = (
    connectors_df.groupby(["operator","connector_type"], dropna=False)
    .agg(total_connectors=("connector_id","count"), max_power_kw=("max_power_kw","max"),
         avg_power_kw=("max_power_kw","mean"))
    .reset_index().sort_values("total_connectors", ascending=False)
)


In [13]:
print("Top 5 provinces by connector count:")
print(summary_province.head(5).to_string(index=False))

Top 5 provinces by connector count:
         province  total_connectors  total_sites  total_operators  max_power_kw  avg_power_kw
        Barcelona              6767         1501               43      400000.0  26427.872026
           Madrid              5744         1399               42      400000.0  41095.987117
Valencia/València              2452          700               32      350000.0  41340.411909
 Alicante/Alacant              2127          652               27     1000000.0  43602.200282
   Balears, Illes              1266          518               35      350000.0  31707.124803


In [14]:
print("Top 5 municipalities by connector count:")
print(summary_municipality.head(5).to_string(index=False))

Top 5 municipalities by connector count:
         province municipality  total_connectors  total_sites  total_operators  max_power_kw  avg_power_kw
           Madrid       Madrid              2721          605               34      400000.0  32292.521132
        Barcelona    Barcelona              3691          579               30      175000.0   9744.475752
          Sevilla      Sevilla               609          206               23      250000.0  37511.330049
Valencia/València     València               659          167               18      175000.0  29137.996965
   Balears, Illes        Palma               379          159               17      350000.0  29076.490765


In [15]:
print("Top 5 autonomous communities by connector count:")
print(summary_autonomous_community.head(5).to_string(index=False))

Top 5 autonomous communities by connector count:
autonomous_community  total_connectors  total_sites  total_operators  max_power_kw  avg_power_kw
            Cataluña              9458         2350               53      400000.0  34274.061112
           Andalucía              5514         1676               43      400000.0  47330.074356
Comunitat Valenciana              5132         1504               39     1000000.0  42976.946610
Madrid, Comunidad de              5744         1399               42      400000.0  41095.987117
     Castilla y León              2769          832               33      400000.0  61541.722644


In [16]:
print("Top 5 operators by connector count:")
print(summary_operator.head(5).to_string(index=False))

Top 5 operators by connector count:
                           operator  total_connectors  total_sites  provinces_covered  municipalities_covered  max_power_kw  avg_power_kw
           IBERDROLA CLIENTES S.A.U             10694         2732                 51                     952      400000.0  32568.464560
   REPSOL SOLUCIONES ENERGÉTICAS SA              6354         1760                 50                     855      560000.0  36672.395342
                 ENDESA X WAY, S.L.              5688         2821                 50                     701      350000.0  38826.420534
BARCELONA DE SERVEIS MUNICIPALS, SA              2598          177                  1                       2       50000.0   6381.524249
               CHARGING TOGETHER SL              2067          482                 48                     312     1000000.0 114295.113691


### Map 2 — Existing EV Charging Stations

Markers are **clustered** for readability at low zoom; individual stations appear on zoom-in.

**Colour coding by max connector power:**
- Red — DC Fast (>= 50 kW)
- Orange — AC High (22–49 kW)
- Blue — AC Standard (< 22 kW)
- Grey — Power unknown

Click any marker for station details (operator, connectors, power, municipality).


In [17]:
# ── GeoDataFrame of sites with valid coordinates ──────────────────────────────
sites_geo = sites_df.dropna(subset=["latitude", "longitude"]).copy()
sites_gdf = gpd.GeoDataFrame(
    sites_geo,
    geometry=gpd.points_from_xy(sites_geo["longitude"], sites_geo["latitude"]),
    crs=CRS_GEO
)

def charger_color(max_kw):
    # Colour by peak connector power tier
    if pd.isna(max_kw): return "gray"
    elif max_kw >= 50:  return "red"
    elif max_kw >= 22:  return "orange"
    else:               return "blue"

# ── Build Map 2 ───────────────────────────────────────────────────────────────
m2 = folium.Map(location=SPAIN_CENTER, zoom_start=DEFAULT_ZOOM, tiles=MAP_TILES)
cluster = MarkerCluster(name="EV Charging Stations").add_to(m2)

for _, row in sites_gdf.iterrows():
    popup_html = (
        f"<b>{row.get('site_name') or 'Station'}</b><br>"
        f"<i>{row.get('operator') or 'Unknown operator'}</i><br><br>"
        f"Location: {row.get('municipality') or '—'}, {row.get('province') or '—'}<br>"
        f"Connectors: <b>{int(row['n_connectors']) if pd.notna(row['n_connectors']) else '—'}</b><br>"
        f"Max power: <b>{row.get('site_max_power_kw') or '—'} kW</b><br>"
        f"Refill points: {int(row['n_refill_points']) if pd.notna(row['n_refill_points']) else '—'}"
    )
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6, color=charger_color(row.get("site_max_power_kw")),
        fill=True, fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f"{row.get('site_name') or 'Station'} | {row.get('site_max_power_kw') or '?'} kW"
    ).add_to(cluster)

legend = (
    '<div style="position:fixed;bottom:40px;left:40px;z-index:9999;background:white;'
    'padding:12px 16px;border-radius:8px;border:1px solid #ccc;font-size:12px;line-height:1.9">'
    '<b>Max Connector Power</b><br>'
    '<span style="color:red">&#9679;</span> DC Fast (&#8805;50 kW)<br>'
    '<span style="color:orange">&#9679;</span> AC High (22&#8211;49 kW)<br>'
    '<span style="color:#4575b4">&#9679;</span> AC Standard (&lt;22 kW)<br>'
    '<span style="color:gray">&#9679;</span> Unknown'
    '</div>'
)
m2.get_root().html.add_child(folium.Element(legend))

m2.save(str(OUT / "map2_ev_chargers.html"))
print(f"Map 2 saved — {len(sites_gdf):,} stations plotted")
IFrame(src=str(OUT / "map2_ev_chargers.html"), width="100%", height=550)


Map 2 saved — 12,075 stations plotted


---
<a id="grid"></a>
## 3. Power Grid Access (i-DE / CNMC Demand Data)

This section analyses **available grid connection capacity** at substations across Spain.

Each row is a **connection point** (substation node) with available MW headroom for new demand connections.

### Viability thresholds
| Tier | Available capacity | Interpretation |
|---|---|---|
| Fast charging viable | >= 1 MW | Supports 6+ x 150 kW chargers without upgrade |
| HPC hub viable | >= 5 MW | Supports 33+ chargers; full corridor hub |

> Source: i-DE (Iberdrola distribución) — CNMC Resolución Capacidad Acceso Demanda 2026.
> File: `2026_04_01_R1-001_Demanda.csv`

### Distributor territories (CNMC)
- **i-DE (Iberdrola)** — Centre, North, Basque Country, Aragon, Castilla, Valencia, Murcia
- **Endesa** — Cataluña, Andalucía, Extremadura
- **UFD (Naturgy)** — Galicia, Madrid, Castilla-La Mancha, Canarias
- **Viesgo (E.ON)** — Cantabria, Asturias


In [25]:
df_ide = pd.read_csv("Data_Supply/2026_04_01_R1-001_Demanda.csv", sep=";")
df_ide.head()

,Gestor de red,Provincia,Municipio,Coordenada UTM X,Coordenada UTM Y,Subestación,Nivel de Tensión (kV),Capacidad firme disponible (MW),Capacidad comprometida por cuestiones regulatorias,Capacidad de acceso firme de demanda ocupada (MW),Capacidad de acceso firme admitida y no evaluada (MW),Posiciones ocupadas,Posiciones libres,Nudo 0*,Comentarios,Denominación del Punto de Conexión,Identificador del Punto de Conexión
0,R1-001,Araba/Álava,Ayala/Aiara,"499100,028809514","4770106,21764367",3102,30,0,0,"60,19",0,NaN,NaN,0,NaN,ST AIALA 30.000,25740000
1,R1-001,Araba/Álava,Vitoria-Gasteiz,"523922,106338171","4744874,00934341",3017,"13,2",0,0,15,0,NaN,NaN,0,NaN,ALI T3,101065005
2,R1-001,Araba/Álava,Vitoria-Gasteiz,"523922,106338171","4744874,00934341",3017,30,0,0,"72,62",0,NaN,NaN,1,NaN,ALI T3 30.000,25230001
3,R1-001,Araba/Álava,Vitoria-Gasteiz,"523922,106338171","4744874,00934341",3017,"13,2",0,0,"10,45",0,NaN,NaN,0,NaN,ALI T4,101065007
4,R1-001,Araba/Álava,Vitoria-Gasteiz,"523922,106338171","4744874,00934341",3017,30,0,0,"72,27",0,NaN,NaN,1,NaN,ALI T4 30.000,25230002


In [26]:
# ── Standardise column names ──────────────────────────────────────────────────
df_ide.columns = [c.strip() for c in df_ide.columns]

rename_map = {
    "Gestor de red": "grid_manager",
    "Provincia": "province",
    "Municipio": "municipality",
    "Coordenada UTM X": "utm_x",
    "Coordenada UTM Y": "utm_y",
    "Subestación": "substation",
    "Nivel de Tensión (kV)": "voltage_kv",
    "Capacidad firme disponible (MW)": "capacity_available_mw",
    "Capacidad comprometida por cuestiones regulatorias": "capacity_regulatory_mw",
    "Capacidad de acceso firme de demanda ocupada (MW)": "capacity_occupied_mw",
    "Capacidad de acceso firme admitida y no evaluada (MW)": "capacity_pending_mw",
    "Posiciones ocupadas": "positions_occupied",
    "Posiciones libres": "positions_free",
    "Nudo 0*": "node_0",
    "Comentarios": "comments",
    "Denominación del Punto de Conexión": "connection_point_name",
    "Identificador del Punto de Conexión": "connection_point_id",
}
df_ide = df_ide.rename(columns=rename_map)


In [27]:
# ── Parse numeric columns: handle Spanish number formatting ──────────────────
# Spanish CSVs often use '.' as thousands separator and ',' as decimal separator
num_cols = [
    "utm_x", "utm_y", "voltage_kv",
    "capacity_available_mw", "capacity_regulatory_mw",
    "capacity_occupied_mw", "capacity_pending_mw",
    "positions_occupied", "positions_free", "node_0", "connection_point_id",
]

for col in num_cols:
    if col in df_ide.columns:
        df_ide[col] = (
            df_ide[col].astype(str).str.strip()
            .str.replace(".", "", regex=False)   # remove thousands separator
            .str.replace(",", ".", regex=False)  # decimal comma -> decimal point
            .replace({"nan": np.nan, "": np.nan})
        )
        df_ide[col] = pd.to_numeric(df_ide[col], errors="coerce")

df_ide.head()


,grid_manager,province,municipality,utm_x,utm_y,substation,voltage_kv,capacity_available_mw,capacity_regulatory_mw,capacity_occupied_mw,capacity_pending_mw,positions_occupied,positions_free,node_0,comments,connection_point_name,connection_point_id
0,R1-001,Araba/Álava,Ayala/Aiara,499100.028810,4.770106e+06,3102,30.0,0.0,0,60.19,0.0,NaN,NaN,0,NaN,ST AIALA 30.000,25740000
1,R1-001,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,3017,13.2,0.0,0,15.00,0.0,NaN,NaN,0,NaN,ALI T3,101065005
2,R1-001,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,3017,30.0,0.0,0,72.62,0.0,NaN,NaN,1,NaN,ALI T3 30.000,25230001
3,R1-001,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,3017,13.2,0.0,0,10.45,0.0,NaN,NaN,0,NaN,ALI T4,101065007
4,R1-001,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,3017,30.0,0.0,0,72.27,0.0,NaN,NaN,1,NaN,ALI T4 30.000,25230002


In [28]:
# ── Build GeoDataFrame — project from ETRS89/UTM30N (EPSG:25830) to WGS84 ───
gdf_ide = gpd.GeoDataFrame(
    df_ide.dropna(subset=["utm_x", "utm_y"]).copy(),
    geometry=gpd.points_from_xy(
        df_ide.dropna(subset=["utm_x", "utm_y"])["utm_x"],
        df_ide.dropna(subset=["utm_x", "utm_y"])["utm_y"]
    ),
    crs="EPSG:25830"
).to_crs(4326)

gdf_ide[["province", "municipality", "utm_x", "utm_y", "geometry"]].head()


,province,municipality,utm_x,utm_y,geometry
0,Araba/Álava,Ayala/Aiara,499100.028810,4.770106e+06,POINT (-3.01106 43.08367)
1,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,POINT (-2.70719 42.85608)
2,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,POINT (-2.70719 42.85608)
3,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,POINT (-2.70719 42.85608)
4,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,POINT (-2.70719 42.85608)


In [29]:
# ── Derived capacity metrics ──────────────────────────────────────────────────

# Total committed capacity (regulatory + occupied + pending requests)
gdf_ide["capacity_constrained_mw"] = (
    gdf_ide["capacity_regulatory_mw"].fillna(0) +
    gdf_ide["capacity_occupied_mw"].fillna(0) +
    gdf_ide["capacity_pending_mw"].fillna(0)
)

# Viability flags for fast-charging deployment
gdf_ide["viable_fast_charging"] = gdf_ide["capacity_available_mw"] >= 1  # >=1 MW: 6+ chargers
gdf_ide["viable_hpc"]           = gdf_ide["capacity_available_mw"] >= 5  # >=5 MW: full hub


In [30]:
# ── Grid capacity summaries ───────────────────────────────────────────────────

summary_grid_province = (
    gdf_ide.groupby("province", dropna=False)
    .agg(n_nodes=("connection_point_id","count"),
         total_available_mw=("capacity_available_mw","sum"),
         avg_available_mw=("capacity_available_mw","mean"),
         max_available_mw=("capacity_available_mw","max"),
         viable_fast_nodes=("viable_fast_charging","sum"),
         viable_hpc_nodes=("viable_hpc","sum"))
    .reset_index().sort_values("total_available_mw", ascending=False)
)

summary_grid_municipality = (
    gdf_ide.groupby(["province","municipality"], dropna=False)
    .agg(n_nodes=("connection_point_id","count"),
         total_available_mw=("capacity_available_mw","sum"),
         avg_available_mw=("capacity_available_mw","mean"),
         max_available_mw=("capacity_available_mw","max"),
         viable_fast_nodes=("viable_fast_charging","sum"),
         viable_hpc_nodes=("viable_hpc","sum"))
    .reset_index().sort_values(["province","total_available_mw"], ascending=[True,False])
)

print("Top 5 provinces by total available grid capacity:")
print(summary_grid_province.head(5).to_string(index=False))


Top 5 provinces by total available grid capacity:
         province  n_nodes  total_available_mw  avg_available_mw  max_available_mw  viable_fast_nodes  viable_hpc_nodes
           Murcia      166              517.21          3.115723             43.71                 28                20
           Madrid      441              515.46          1.168844             29.43                 49                34
Valencia/València      215              509.25          2.368605             63.15                 26                18
           Zamora       62              336.08          5.420645            112.34                 13                10
          Navarra      238              332.18          1.395714             29.56                 22                17


### Map 3 — Grid Connection Capacity

Each circle is a connection point (substation node). **Size and colour** reflect available MW for new demand connections.

- Red (large) — > 20 MW — abundant capacity, ideal for HPC hubs
- Orange — 5–20 MW — medium hub viable
- Green — 1–5 MW — standard fast-charging viable
- Blue (small) — < 1 MW — constrained; upgrade required

Click any marker for full substation details.


In [31]:
import folium
from folium.plugins import MarkerCluster

def capacity_color(mw):
    if pd.isna(mw): return "gray"
    elif mw < 1:    return "blue"
    elif mw < 5:    return "green"
    elif mw < 20:   return "orange"
    else:           return "red"

def capacity_radius(mw):
    if pd.isna(mw): return 3
    elif mw < 1:    return 3
    elif mw < 5:    return 5
    elif mw < 20:   return 7
    else:           return 9

m_grid = folium.Map(location=SPAIN_CENTER, zoom_start=DEFAULT_ZOOM, tiles=MAP_TILES)
cluster = MarkerCluster().add_to(m_grid)

for _, row in gdf_ide.iterrows():
    popup_html = (
        f"<b>{row.get('connection_point_name','—')}</b><br>"
        f"Grid manager: {row.get('grid_manager','—')}<br>"
        f"Province: {row.get('province','—')}<br>"
        f"Municipality: {row.get('municipality','—')}<br>"
        f"Substation: {row.get('substation','—')}<br>"
        f"Voltage: {row.get('voltage_kv','—')} kV<br>"
        f"Available capacity: <b>{row.get('capacity_available_mw','—')} MW</b><br>"
        f"Occupied: {row.get('capacity_occupied_mw','—')} MW | "
        f"Pending: {row.get('capacity_pending_mw','—')} MW<br>"
        f"Free positions: {row.get('positions_free','—')} | "
        f"Occupied positions: {row.get('positions_occupied','—')}"
    )
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=capacity_radius(row.get("capacity_available_mw")),
        color=capacity_color(row.get("capacity_available_mw")),
        fill=True, fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=320),
        tooltip=f"{row.get('connection_point_name','—')} | {row.get('capacity_available_mw','—')} MW"
    ).add_to(cluster)

legend = (
    '<div style="position:fixed;bottom:40px;left:40px;z-index:9999;background:white;'
    'padding:12px 16px;border-radius:8px;border:1px solid #ccc;font-size:12px;line-height:1.9">'
    '<b>Available Grid Capacity</b><br>'
    '<span style="color:red">&#9679;</span> &gt;20 MW (HPC hub viable)<br>'
    '<span style="color:orange">&#9679;</span> 5&#8211;20 MW (medium hub)<br>'
    '<span style="color:green">&#9679;</span> 1&#8211;5 MW (fast charging viable)<br>'
    '<span style="color:#4575b4">&#9679;</span> &lt;1 MW (constrained)<br>'
    '<span style="color:gray">&#9679;</span> No data'
    '</div>'
)
m_grid.get_root().html.add_child(folium.Element(legend))

m_grid.save(str(OUT / "map3_grid_capacity.html"))
print("Map 3 saved")
IFrame(src=str(OUT / "map3_grid_capacity.html"), width="100%", height=550)


Map 3 saved


---
<a id="combined"></a>
## 4. Combined Analysis Map

This final map integrates all four data layers into a single interactive view with **toggleable layer controls**.

### Layers available
| Toggle label | What it shows | Use for |
|---|---|---|
| Traffic High | > 20k travellers/day | Priority deployment corridors |
| Traffic Medium | 5–20k/day | Secondary coverage targets |
| Traffic Low | < 5k/day | AFIR gap compliance check |
| Existing Chargers | All current stations | Coverage gap identification |
| Grid HPC viable | >= 5 MW nodes | Optimal hub locations |
| Grid Fast viable | 1–5 MW nodes | Standard station sites |
| Grid Constrained | < 1 MW nodes | Upgrade-required locations |

### Analysis workflow
1. Enable **Traffic High** + **Existing Chargers** to identify high-demand gaps in coverage
2. Enable **Grid HPC viable** on top to find actionable deployment opportunities
3. Zoom into specific corridors to evaluate co-location of grid nodes and traffic gaps


In [34]:
def traffic_color(total):
    if pd.isna(total): return "#cccccc"
    elif total >= 50_000: return "#d73027"
    elif total >= 20_000: return "#fc8d59"
    else:                 return "#fee090"

m_combined = folium.Map(location=SPAIN_CENTER, zoom_start=DEFAULT_ZOOM, tiles=MAP_TILES)

# ── Traffic — only roads >= 20k/day, aggressively simplified ─────────────────
traffic_filtered = traffic_map[traffic_map["Total"] >= 20_000].copy()
traffic_filtered = traffic_filtered.to_crs(3042)
traffic_filtered["geometry"] = traffic_filtered.geometry.simplify(200, preserve_topology=True)
traffic_filtered = traffic_filtered.to_crs(4326)

fg_traffic_high = folium.FeatureGroup(name="Traffic >50k/day", show=True)
fg_traffic_med  = folium.FeatureGroup(name="Traffic 20-50k/day", show=True)

for _, row in traffic_filtered.iterrows():
    total = row.get("Total")
    if pd.isna(total): continue
    geom  = row.geometry
    lines = geom.geoms if geom.geom_type == "MultiLineString" else [geom]
    color = traffic_color(total)
    target = fg_traffic_high if total >= 50_000 else fg_traffic_med
    for line in lines:
        coords = [(lat, lon) for lon, lat in line.coords]
        if len(coords) < 2: continue
        folium.PolyLine(coords, color=color, weight=2.5, opacity=0.75).add_to(target)

for fg in [fg_traffic_high, fg_traffic_med]:
    fg.add_to(m_combined)

# ── Existing chargers — heatmap (much lighter than 12k individual markers) ────
fg_chargers = folium.FeatureGroup(name="Charger Density (heatmap)", show=True)
heat_data = [
    [row.geometry.y, row.geometry.x]
    for _, row in sites_gdf.iterrows()
]
HeatMap(heat_data, radius=12, blur=15, min_opacity=0.4).add_to(fg_chargers)
fg_chargers.add_to(m_combined)

# ── Grid capacity — HPC and fast nodes only (drop constrained) ────────────────
fg_grid_hpc  = folium.FeatureGroup(name="Grid HPC viable >=5 MW", show=True)
fg_grid_fast = folium.FeatureGroup(name="Grid Fast viable 1-5 MW", show=False)

for _, row in gdf_ide.iterrows():
    mw = row.get("capacity_available_mw")
    if pd.isna(mw) or mw < 1: continue
    popup_html = (
        f"<b>{row.get('connection_point_name','—')}</b><br>"
        f"{row.get('municipality','—')}, {row.get('province','—')}<br>"
        f"Available: <b>{mw} MW</b>"
    )
    if mw >= 5:  target, color, r = fg_grid_hpc, "#2ca02c", 8
    else:        target, color, r = fg_grid_fast, "#ff7f0e", 5

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=r, color=color, fill=True, fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=f"{mw} MW"
    ).add_to(target)

for fg in [fg_grid_hpc, fg_grid_fast]:
    fg.add_to(m_combined)

# ── Layer control + legend ────────────────────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(m_combined)

legend = (
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;'
    'padding:14px 18px;border-radius:8px;border:1px solid #ccc;'
    'font-size:11px;line-height:2;max-width:260px">'
    '<b style="font-size:12px">Combined Legend</b><br>'
    '<b>Traffic (travellers/day)</b><br>'
    '<span style="color:#d73027">&#9472;&#9472;</span> &gt;50k '
    '<span style="color:#fc8d59">&#9472;&#9472;</span> 20&#8211;50k<br>'
    '<b>Grid Capacity (MW available)</b><br>'
    '<span style="color:#2ca02c">&#9679;</span> &#8805;5 MW (HPC) '
    '<span style="color:#ff7f0e">&#9679;</span> 1&#8211;5 MW<br>'
    '<b>Chargers</b> — heatmap (density)'
    '</div>'
)
m_combined.get_root().html.add_child(folium.Element(legend))

m_combined.save(str(OUT / "map_combined_2.html"))
print(f"Saved → map_combined_2.html")
IFrame(src=str(OUT / "map_combined_2.html"), width="100%", height=620)


Saved → map_combined_2.html


---
## Appendix — Data Sources & Regulatory References

### Data Sources

| Dataset | Provider | Licence | URL |
|---|---|---|---|
| Road network (OSM) | OpenStreetMap contributors | ODbL | https://www.openstreetmap.org |
| Traffic flows (BigData Movilidad) | Ministerio de Transportes | Open data | https://mapas.fomento.gob.es |
| EV charging stations (DATEX II) | DGT | Open data | https://infocar.dgt.es |
| Grid demand capacity | i-DE / CNMC | Regulatory publication | https://www.cnmc.es |

### Regulatory Framework

- **EU AFIR Regulation 2023/1804** — HPC chargers required every 60 km on TEN-T Core corridors and every 100 km on Comprehensive corridors by 2025–2030.
- **Royal Decree 569/2020** — Spanish transposition framework for EV infrastructure.
- **CNMC Resolution** — grid access capacity rules for demand connection points.

### Known Limitations

1. **Traffic data** reflects 2023 mobility patterns; EV charging demand is growing ~15%/year per DGT projections.
2. **Grid capacity** covers i-DE (Iberdrola) territory only. Endesa, UFD, and Viesgo zones require separate CNMC datasets.
3. **Charger power** values are self-reported; operational availability is not tracked in the DGT feed.
4. **AFIR gap analysis** (150 km rule) not yet implemented — requires network-graph routing along OSM road segments.
